In [9]:
from losses import completion_network_loss, noise_loss
from utils import *
from classify import *
from generator import *
from discri import *
from torch.utils.data import DataLoader
from torch.optim import Adadelta, Adam
from torch.nn import BCELoss, DataParallel
from torchvision.utils import save_image
from torch.autograd import grad
import torchvision.transforms as transforms
import torch
import time
import random
import os, logging
import numpy as np
from attack import inversion, dist_inversion
from generator import Generator
from argparse import ArgumentDefaultsHelpFormatter, ArgumentParser
from torch.autograd import Variable

# Load models
z_dim = 100
G = Generator(z_dim)
G = torch.nn.DataParallel(G).cuda()
path_G = './improvedGAN/improved_celeba_G.tar'
ckp_G = torch.load(path_G)
G.load_state_dict(ckp_G['state_dict'], strict=False)

T = VGG16(1000)
path_T = './target_model/target_ckp/VGG16_88.26.tar'
T = torch.nn.DataParallel(T).cuda()
ckp_T = torch.load(path_T)
T.load_state_dict(ckp_T['state_dict'], strict=False)


# initalize a sample of recovery script:
iden = torch.from_numpy(np.arange(60))
iden = iden.view(-1).long().cuda()

# get evaluation metric for backprop:
criterion = nn.CrossEntropyLoss().cuda()

bs = iden.shape[0]
no = torch.zeros(bs) # index for saving all success attack images
mu = Variable(torch.zeros(bs, 100), requires_grad=True)
log_var = Variable(torch.ones(bs, 100), requires_grad=True)

G.eval()
T.eval()

def reparameterize(mu, logvar):
    """
    Reparameterization trick to sample from N(mu, var) from
    N(0,1).
    :param mu: (Tensor) Mean of the latent Gaussian [B x D]
    :param logvar: (Tensor) Standard deviation of the latent Gaussian [B x D]
    :return: (Tensor) [B x D]
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)

    return eps * std + mu



In [12]:
z = reparameterize(mu, log_var)
fake = G(z)
out = T(fake)[-1]

In [13]:
out

tensor([[-3.7772,  3.0906, -3.1957,  ...,  3.0673,  5.5927,  4.2924],
        [ 1.2891,  0.3919, -0.4277,  ..., -5.1423, -4.9857, -1.5747],
        [ 8.8576, -0.8270, -6.1675,  ...,  0.1386, -4.3905, -2.8617],
        ...,
        [-1.2502, -0.7041,  6.2134,  ..., -5.8178, -1.9798,  0.5596],
        [-1.0964,  2.5321,  4.7329,  ..., -2.6736, -0.4741,  0.3272],
        [ 2.5223,  0.8366,  0.6483,  ..., -4.1661, -3.7407,  3.7747]],
       device='cuda:0', grad_fn=<AddmmBackward0>)